# Stage 3 — Disaggregate households into five income groups

Restricted 2024 HBS records allocate the public 2023 SAM household account across population quintiles. Microdata remain in an external access-controlled directory and are never exported by this notebook.

## 1. Locations, parameters and input checks

Set HBS_DATA_DIR to the authorised survey directory. The manifest verifies the public SAM and the required restricted source files. All macro amounts use thousand TRY.

In [ ]:
from pathlib import Path
import json, hashlib, math, os, sys, tempfile, subprocess, shutil, platform
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from IPython.display import display

def locate_stage():
    here=Path.cwd().resolve()
    for parent in [here,*here.parents]:
        for candidate in (parent,parent/'3_household_groups'):
            if (candidate/'datas/input_manifest.json').is_file() and (candidate/'codes').is_dir():
                return candidate
    raise FileNotFoundError('3_household_groups/datas not found; open the notebook from its codes directory.')

ROOT=locate_stage(); DATA=ROOT/'datas'; OUT=ROOT/'outputs'
MICRO_DIR=Path(os.environ.get('HBS_DATA_DIR',DATA)).expanduser().resolve()
OUT.mkdir(exist_ok=True)
GROUPS=[f'Q{i}' for i in range(1,6)]
SECTORS=[f'S{i:02d}' for i in range(64)]
MIXED_LABOR_SHARE=0.50
SUPPORT_PRIOR=1e-8
RAS_TOL=1e-14
BALANCE_ATOL=0.001  # thousand TRY = 1 TL
assert 0 <= MIXED_LABOR_SHARE <= 1
assert 0 < SUPPORT_PRIOR < 1
def sha256(path):
    d=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):d.update(block)
    return d.hexdigest()

manifest=json.loads((DATA/'input_manifest.json').read_text(encoding='utf-8'))
for entry in manifest['files']:
    assert sha256((DATA if entry['file'].endswith('.xlsx') else MICRO_DIR)/entry['file'])==entry['sha256'], 'Input checksum changed: '+entry['file']
checks=[]
def verify(name,actual,expected,atol=BALANCE_ATOL):
    a,b=np.asarray(actual,dtype=float),np.asarray(expected,dtype=float)
    delta=float(np.max(np.abs(a-b)))
    ok=bool(np.isfinite(a).all() and np.isfinite(b).all() and delta<=atol)
    checks.append(dict(check=name,max_absolute_difference=delta,tolerance=atol,passed=ok))
    assert ok, f'{name}: difference={delta}, tolerance={atol}'
print('Five inputs verified. SAM: 2023; HBS: 2024; SAM unit: thousand TRY.')

## 2. Read SAM and restricted HBS inputs

The 73-account SAM is read from SAM_73. Household, person and consumption records are linked using the original TurkStat identifiers; the item dictionary validates COICOP codes. These source variable identifiers are retained for compatibility.

In [ ]:
source_path=DATA/'SAM_2023_64_sectors.xlsx'
source=pd.read_excel(source_path,sheet_name='SAM_73',header=4,index_col=0).iloc[:73,:73].astype(float)
source_accounts=SECTORS+['LAB','CAP','PTAX','CTAX','DTAX','HOH','GOV','INV','EXT']
assert list(source.index)==list(source.columns)==source_accounts
assert np.isfinite(source.to_numpy()).all()
verify('Source SAM row-column balances',source.sum(axis=1),source.sum(axis=0))
accounts_source=pd.read_excel(source_path,sheet_name='Accounts',header=4).iloc[:73].set_index('Account')
for a in source_accounts:
    if a not in ['LAB','CAP']:assert source.loc['HOH',a]==0, ('Unrecognised household income',a)
    if a not in SECTORS+['CTAX','DTAX','INV']:assert source.loc[a,'HOH']==0, ('Unrecognised household payment',a)
assert (source.loc[SECTORS,'HOH']>0).all()

h=pd.read_csv(MICRO_DIR/'hane_cd_2024.csv',sep=';').copy()
f=pd.read_csv(MICRO_DIR/'fert_cd_2024.csv',sep=';')
t=pd.read_csv(MICRO_DIR/'tuketim_cd_2024.csv',sep=';')
dictionary=pd.read_csv(MICRO_DIR/'hbs_madde_tanim.csv',sep=';',encoding='cp1254')
assert h.BIRIMNO.is_unique
assert not f.duplicated(['BIRIMNO','FERTNO']).any()
assert f.BIRIMNO.isin(h.BIRIMNO).all() and t.BIRIMNO.isin(h.BIRIMNO).all()
core=['YILKULGEL','EFB_OECD','FAKTOR','HHB','HARCAMA']
assert np.isfinite(h[core].to_numpy(float)).all()
assert (h[['EFB_OECD','FAKTOR','HHB','HARCAMA']]>0).all().all()
assert (h.YILKULGEL>=0).all()
assert np.isfinite(t.DEGER.to_numpy(float)).all() and (t.DEGER>=0).all()
roster=f.groupby('BIRIMNO').size().reindex(h.BIRIMNO).to_numpy()
verify('Person roster matches household size',roster,h.HHB.to_numpy(),atol=0)
expense=t.groupby('BIRIMNO').DEGER.sum().reindex(h.BIRIMNO).to_numpy()
verify('Consumption records match HARCAMA totals (TRY)',expense,h.HARCAMA.to_numpy(),atol=.1)
print(f'{len(h):,} households, {len(f):,} persons, {len(t):,} consumption records read.')

## 3. Population-weighted income quintiles

Equivalent income is $y_j=Y_j/e_j$ and person weight is $v_j=w_j n_j$. Households are sorted by equivalent income and assigned using cumulative weighted midpoints; a household is never split.

In [ ]:
h['equivalent_income']=h.YILKULGEL/h.EFB_OECD
h['person_weight']=h.FAKTOR*h.HHB
h=h.sort_values(['equivalent_income','BIRIMNO'],kind='stable').reset_index(drop=True)
mid=(h.person_weight.cumsum()-h.person_weight/2)/h.person_weight.sum()
h['group']=[f'Q{min(5,int(x*5)+1)}' for x in mid]
assert h.group.nunique()==5
assert h.equivalent_income.is_monotonic_increasing
verify('Groups preserve the represented population',h.groupby('group').person_weight.sum().sum(),h.person_weight.sum(),atol=1e-5)
print('Q1 is the lowest and Q5 the highest equivalent-income group.')
print('Retained zero-income households:',int((h.YILKULGEL==0).sum()))

## 4. Factor-income allocation proxies

Indexed wages and half of mixed income define the labour proxy. Financial and rental income, annual imputed rent and the remaining mixed income define the capital proxy. Negative proxy components are floored at zero before group aggregation.

In [ ]:
income_index=pd.to_numeric(f.INDEKS_ARL,errors='raise')
assert np.isfinite(income_index).all() and (income_index>0).all()
assert f.groupby('BIRIMNO').INDEKS_ARL.nunique().max()==1
def person_sum(columns):
    v=f[columns].fillna(0).sum(axis=1)*income_index
    return v.groupby(f.BIRIMNO).sum().reindex(h.BIRIMNO).to_numpy()
wage=person_sum(['GELIR_NAKDI_UCRET_YEVMIYE','GELIR_NAKDI_IKRAMIYE_PRIM','GELIR_AYNI_UCRET_YEVMIYE'])
mixed=person_sum(['GELIR_NAKDI_MUTESEBBIS','GELIR_AYNI_MUTESEBBIS','GELIR_TARIM_NET'])
financial=person_sum(['GELIR_BANKA_TAHVIL','GELIR_BIREYSEL','GELIR_DIGER_MENKUL'])
hindex=f.groupby('BIRIMNO').INDEKS_ARL.first().reindex(h.BIRIMNO).to_numpy()
rent=(h.GELIR_NAKDI_GAYRIMENKUL.fillna(0)+h.GELIR_AYNI_GAYRIMENKUL.fillna(0)).to_numpy()*hindex
imputed=12*h.KIRA_IZAFI.fillna(0).to_numpy()
labor_raw=wage+MIXED_LABOR_SHARE*mixed
capital_raw=financial+rent+imputed+(1-MIXED_LABOR_SHARE)*mixed
h['labor_proxy']=np.maximum(labor_raw,0)
h['capital_proxy']=np.maximum(capital_raw,0)
rows=[]
for q in GROUPS:
    sub=h[h.group==q];w=sub.FAKTOR
    rows.append(dict(group=q,sample_households=len(sub),weighted_households=float(w.sum()),
        population=float(sub.person_weight.sum()),income_total_tl=float((sub.YILKULGEL*w).sum()),
        labor_proxy_tl=float((sub.labor_proxy*w).sum()),capital_proxy_tl=float((sub.capital_proxy*w).sum()),
        consumption_annualized_tl=float((12*sub.HARCAMA*w).sum()),
        equivalent_income_min_tl=float(sub.equivalent_income.min()),equivalent_income_max_tl=float(sub.equivalent_income.max())))
profiles=pd.DataFrame(rows).set_index('group').loc[GROUPS]
for share,total in [('population_share','population'),('income_share','income_total_tl'),
                    ('consumption_share','consumption_annualized_tl'),('labor_share','labor_proxy_tl'),('capital_share','capital_proxy_tl')]:
    assert profiles[total].sum()>0
    profiles[share]=profiles[total]/profiles[total].sum()
    verify(share+' sum',profiles[share].sum(),1,atol=1e-12)
display(profiles[['sample_households','population_share','income_share','consumption_share','labor_share','capital_share']].round(6))

## 5. COICOP-to-industry consumption bridge

Monthly weighted expenditure is annualised and aggregated by COICOP and quintile. Each industry uses an explicit basket; broad consumption shares provide a documented fallback. A small positive support prior permits RAS balancing.

In [ ]:
SECTOR_BASKETS = {'S00': '011', 'S01': '0454', 'S02': '0113', 'S03': '0454', 'S04': '01|02', 'S05': '03|052', 'S06': '0431|051', 'S07': '0561|0974', 'S08': '097', 'S09': '0453|0722', 'S10': '0561|1312', 'S11': '0611', 'S12': '054|0721', 'S13': '0431|054', 'S14': '0431|054', 'S15': '0431|054|055', 'S16': '081|0911', 'S17': '053|1311', 'S18': '055', 'S19': '0711|09121', 'S20': '0712|0713|09122|09123', 'S21': '051|092|0951|1321', 'S22': '0533|055|0944', 'S23': '0451|0452|0455', 'S24': '0441', 'S25': '0442|0443', 'S26': '043', 'S27': '0721|0723', 'S28': '01|02|03|05|061|071|081|091|092|093|095|097|131|132', 'S29': '01|02|03|05|061|071|081|091|092|093|095|097|131|132', 'S30': '0731|0732|0735', 'S31': '0734', 'S32': '0733', 'S33': '0724|0736|0749', 'S34': '0741', 'S35': '11', 'S36': '0971|0972|0973|082', 'S37': '0839|0952|0961', 'S38': '0831|0832|0833|0834', 'S39': '082|0833|08399', 'S40': '122', 'S41': '121', 'S42': '122', 'S43': '041', 'S44': '042', 'S45': '139', 'S46': '043', 'S47': '*', 'S48': '*', 'S49': '0945|0963', 'S50': '0941|0942|0943|0944', 'S51': '*', 'S52': '098', 'S53': '0444|0562', 'S54': '*', 'S55': '10', 'S56': '062|063|064', 'S57': '133', 'S58': '0947|096', 'S59': '0946', 'S60': '13902', 'S61': '0314|0322|0533|0835|0941|0942|0943|0944|13112|13212|13292', 'S62': '1313|139', 'S63': '05621'}

In [ ]:
t['coicop']=t.HBS_KOD5.astype(int).astype(str).str.zfill(5)
t=t.merge(h[['BIRIMNO','group','FAKTOR']],on='BIRIMNO',validate='many_to_one')
t['annual_weighted_tl']=12*t.DEGER*t.FAKTOR
coicop=t.pivot_table(index='coicop',columns='group',values='annual_weighted_tl',aggfunc='sum',fill_value=0).reindex(columns=GROUPS)
label_codes=dictionary.iloc[:,1].astype(str).str.replace('.','',regex=False).str.zfill(5)
assert set(coicop.index).issubset(set(label_codes))
verify('COICOP totals match group consumption (TRY)',coicop.sum(axis=0),profiles.consumption_annualized_tl,atol=.1)
cshares=profiles.consumption_share.to_numpy()
row_targets=source.loc[SECTORS,'HOH'].to_numpy()
seed=[];mapping=[]
for i,s in enumerate(SECTORS):
    selector=SECTOR_BASKETS[s]
    mask=np.ones(len(coicop),dtype=bool) if selector=='*' else coicop.index.str.startswith(tuple(selector.split('|')))
    basket=coicop.loc[mask].sum(axis=0).to_numpy()
    fallback=bool(basket.sum()<=0)
    shares=cshares if fallback else basket/basket.sum()
    shares=(1-SUPPORT_PRIOR)*shares+SUPPORT_PRIOR*cshares
    seed.append(row_targets[i]*shares)
    mapping.append(dict(account=s,nace=str(accounts_source.loc[s,'NACE']),coicop_prefixes=selector,
        proxy_type='Total consumption' if selector=='*' else 'Sector basket',matched_codes=int(mask.sum()),
        fallback=fallback,source_consumption_bin_tl=float(row_targets[i])))
seed=np.asarray(seed)
mapping=pd.DataFrame(mapping).set_index('account')
print('Sector:',len(mapping),'COICOP codes:',len(coicop))
print('Sectors using a total-consumption proxy:',mapping.index[mapping.coicop_prefixes=='*'].tolist())
print('Sectors falling back to total-consumption shares:',mapping.index[mapping.fallback].tolist())

## 6. Preserve consumption margins using RAS

Iterative row and column scaling preserves the SAM industry totals and HBS group consumption shares. The tolerance, convergence count and adjustment multipliers are recorded.

In [ ]:
col_targets=row_targets.sum()*cshares
consumption=seed.copy()
for ras_iterations in range(1,10001):
    consumption*= (row_targets/consumption.sum(axis=1))[:,None]
    consumption*= (col_targets/consumption.sum(axis=0))[None,:]
    ras_error=max(float(np.max(abs(consumption.sum(axis=1)/row_targets-1))),float(np.max(abs(consumption.sum(axis=0)/col_targets-1))))
    if ras_error<RAS_TOL:break
else:raise RuntimeError('RAS did not converge.')
verify('RAS sector consumption margins',consumption.sum(axis=1),row_targets)
verify('RAS group consumption margins',consumption.sum(axis=0),col_targets)
mapping['ras_multiplier_min']=(consumption/seed).min(axis=1)
mapping['ras_multiplier_max']=(consumption/seed).max(axis=1)
profiles['consumption_goods0']=consumption.sum(axis=0)
profiles['product_tax0']=float(source.loc['CTAX','HOH'])*profiles.consumption_share
profiles['consumption_budget0']=profiles.consumption_goods0+profiles.product_tax0
verify('Total tax-inclusive consumption',profiles.consumption_budget0.sum(),source.loc[SECTORS+['CTAX'],'HOH'].sum())
print(f'RAS: {ras_iterations} iterations; relative margin error {ras_error:.3e}.')

## 7. Income, direct taxes, reconciliation and saving

Labour and capital totals follow their proxy shares; income and direct tax follow HBS income shares. The zero-sum HH_REC account reconciles these allocations. Saving is disposable income less tax-inclusive consumption, including signed values.

In [ ]:
L0=float(source.loc['HOH','LAB']); K0=float(source.loc['HOH','CAP']); Y0=L0+K0
TD0=float(source.loc['DTAX','HOH'])
profiles['labor0']=L0*profiles.labor_share
profiles['capital0']=K0*profiles.capital_share
profiles['income0']=Y0*profiles.income_share
profiles['direct_tax0']=TD0*profiles.income_share
profiles['reconciliation0']=profiles.income0-profiles.labor0-profiles.capital0
profiles['disposable_income0']=profiles.income0-profiles.direct_tax0
profiles['saving0']=profiles.disposable_income0-profiles.consumption_budget0
profiles['direct_tax_rate0']=profiles.direct_tax0/profiles.income0
profiles['saving_rate_disposable0']=profiles.saving0/profiles.disposable_income0
income_bridge_tl_to_sam=(Y0-TD0)/profiles.income_total_tl.sum()
consumption_bridge_tl_to_sam=profiles.consumption_budget0.sum()/profiles.consumption_annualized_tl.sum()
verify('Total labour income',profiles.labor0.sum(),L0)
verify('Total capital income',profiles.capital0.sum(),K0)
verify('Net income reconciliation total',profiles.reconciliation0.sum(),0)
verify('Total direct tax',profiles.direct_tax0.sum(),TD0)
verify('Source private saving total',profiles.saving0.sum(),source.loc['INV','HOH'])
verify('HBS disposable-income share bridge',profiles.disposable_income0,profiles.income_total_tl*income_bridge_tl_to_sam)
display(profiles[['labor0','capital0','reconciliation0','disposable_income0','consumption_budget0','saving0']].round(3))

## 8. Expanded SAM and reaggregation tests

Q1–Q5 replace HOH and HH_REC is added, giving 78 accounts. Non-household source cells remain unchanged. Collapsing the five groups recovers the original SAM within the accounting tolerance.

In [ ]:
other=[a for a in source_accounts if a!='HOH']
accounts=SECTORS+['LAB','CAP','PTAX','CTAX','DTAX']+GROUPS+['GOV','INV','EXT','HH_REC']
sam=source.loc[other,other].reindex(index=accounts,columns=accounts,fill_value=0.).copy()
for k,q in enumerate(GROUPS):
    sam.loc[SECTORS,q]=consumption[:,k]
    sam.loc['CTAX',q]=profiles.loc[q,'product_tax0']
    sam.loc['DTAX',q]=profiles.loc[q,'direct_tax0']
    sam.loc['INV',q]=profiles.loc[q,'saving0']
    sam.loc[q,'LAB']=profiles.loc[q,'labor0']
    sam.loc[q,'CAP']=profiles.loc[q,'capital0']
    r=float(profiles.loc[q,'reconciliation0'])
    sam.loc[q,'HH_REC']=max(r,0.)
    sam.loc['HH_REC',q]=max(-r,0.)
assert sam.shape==(78,78) and list(sam.index)==list(sam.columns)
verify('Final SAM: 78 account balances',sam.sum(axis=1),sam.sum(axis=0))
verify('Non-household source cells preserved',sam.loc[other,other],source.loc[other,other],atol=0)
collapsed=pd.DataFrame(0.,index=source_accounts,columns=source_accounts)
collapsed.loc[other,other]=sam.loc[other,other]
collapsed.loc[other,'HOH']=sam.loc[other,GROUPS].sum(axis=1)
collapsed.loc['HOH',other]=sam.loc[GROUPS,other].sum(axis=0)
collapsed.loc['HOH','HOH']=sam.loc[GROUPS,GROUPS].to_numpy().sum()
verify('Group aggregation recovers the source SAM',collapsed,source)
verify('Net income recovered from SAM cells',
       sam.loc[GROUPS,['LAB','CAP','HH_REC']].sum(axis=1)-sam.loc['HH_REC',GROUPS],profiles.income0)
verify('Investment financing preserved',sam.loc['INV',GROUPS+['GOV','EXT']].sum(),source['INV'].sum())
assert (sam.loc['HH_REC',GROUPS]>=0).all() and (sam.loc[GROUPS,'HH_REC']>=0).all()
balance=pd.DataFrame({'row_total':sam.sum(axis=1),'column_total':sam.sum(axis=0)})
balance['difference']=balance.row_total-balance.column_total
print('Final SAM:',sam.shape,'| Maximum balance gap (TRY):',balance.difference.abs().max()*1000)
print('Groups with negative saving:',profiles.index[profiles.saving0<0].tolist())

## 9. Prepare aggregate outputs

The workbook includes the full SAM, group summary, consumption bridge, account dictionary, balance checks and metadata. Outputs contain aggregate group statistics only.

In [ ]:
def xlcol(n):return get_column_letter(n)
positions={a:i for i,a in enumerate(accounts)}
def samcell(r,c):return f"'SAM_78'!{xlcol(positions[c]+2)}{positions[r]+2}"
def samrange(rows,c):return f"{samcell(rows[0],c)}:{xlcol(positions[c]+2)}{positions[rows[-1]]+2}"
expected=[]
def frame_spec(name,frame,widths=None):
    a=frame.reset_index()
    values=[list(a.columns)]+a.astype(object).where(pd.notna(a),None).values.tolist()
    return dict(name=name,values=values,widths=widths or {},formulas=[],formats=[],preview='A1:H12')
def add_formula(spec,cell,formula,value=None):
    spec['formulas'].append(dict(cell=cell,formula=formula))
    if value is not None:expected.append(dict(sheet=spec['name'],cell=cell,value=float(value)))

matrix=sam.copy();matrix['TOTAL']=matrix.sum(axis=1);matrix.loc['TOTAL']=matrix.sum(axis=0)
matrix.index.name='Account'
main=frame_spec('SAM_78',matrix,{0:19})
main['preview']='BM65:CB80'
main['numeric_range']='B2:CB80'
main['formats']=[dict(range='A80:CB80',format=dict(fill='#DCE6EE',font=dict(bold=True))),dict(range='CB1:CB80',format=dict(fill='#DCE6EE',font=dict(bold=True,color='#172B4D')))]
for a in accounts:
    rr=positions[a]+2;cc=xlcol(positions[a]+2)
    add_formula(main,f'CB{rr}',f'=SUM(B{rr}:CA{rr})',sam.loc[a].sum())
    add_formula(main,f'{cc}80',f'=SUM({cc}2:{cc}79)',sam[a].sum())
add_formula(main,'CB80','=SUM(B80:CA80)',sam.to_numpy().sum())

summary_values=[['Indicator','Unit']+GROUPS+['Total']]
summary=dict(name='Group_Summary',values=summary_values,widths={0:45,1:13},formulas=[],formats=[],preview='A1:H13')
metric_rows={}
def summary_metric(key,label,unit,vals,fmt='#,##0.00',formulas=None):
    rr=len(summary_values)+1;metric_rows[key]=rr
    summary_values.append([label,unit]+list(map(float,vals))+[None])
    summary['formats'].append(dict(range=f'C{rr}:H{rr}',numberFormat=fmt))
    if formulas:
        for k,formula in enumerate(formulas):add_formula(summary,f'{xlcol(k+3)}{rr}',formula,vals[k])
    add_formula(summary,f'H{rr}',f'=SUM(C{rr}:G{rr})',sum(vals))

for key,label,unit,fmt in [('sample_households','Sample households','households','#,##0'),('weighted_households','Represented households','households','#,##0'),('population','Represented population','persons','#,##0'),('population_share','Population share','share','0.00%'),('income_share','HBS disposable income share','share','0.00%'),('consumption_share','HBS consumption share','share','0.00%'),('labor_share','Labour income allocation share','share','0.00%'),('capital_share','Capital income allocation share','share','0.00%'),('income_total_tl','HBS annual disposable income','TL','#,##0'),('consumption_annualized_tl','HBS annualised consumption','TL','#,##0')]:
    summary_metric(key,label,unit,profiles[key].tolist(),fmt)
macro=[('labor0','Calibrated labour income',lambda q:'='+samcell(q,'LAB')),
       ('capital0','Calibrated capital income',lambda q:'='+samcell(q,'CAP')),
       ('reconciliation0','Net income reconciliation',lambda q:'='+samcell(q,'HH_REC')+'-'+samcell('HH_REC',q)),
       ('income0','Reconciled pre-tax income',lambda q:'='+samcell(q,'LAB')+'+'+samcell(q,'CAP')+'+'+samcell(q,'HH_REC')+'-'+samcell('HH_REC',q)),
       ('direct_tax0','Direct tax',lambda q:'='+samcell('DTAX',q)),
       ('disposable_income0','Calibrated disposable income',lambda q:'='+samcell(q,'LAB')+'+'+samcell(q,'CAP')+'+'+samcell(q,'HH_REC')+'-'+samcell('HH_REC',q)+'-'+samcell('DTAX',q)),
       ('consumption_goods0','Consumption of goods and services',lambda q:'=SUM('+samrange(SECTORS,q)+')'),
       ('product_tax0','Product taxes on final consumption',lambda q:'='+samcell('CTAX',q)),
       ('consumption_budget0','Tax-inclusive consumption budget',lambda q:'=SUM('+samrange(SECTORS,q)+')+'+samcell('CTAX',q)),
       ('saving0','Residual saving',lambda q:'='+samcell('INV',q))]
for key,label,fun in macro:summary_metric(key,label,'thousand TRY',profiles[key].tolist(),formulas=[fun(q) for q in GROUPS])
summary['formats'].append(dict(range='A12:H12',format=dict(borders=dict(top=dict(style='thin',color='#B4C2CF')))))
summary['formats'].append(dict(range=f"H{metric_rows['reconciliation0']}",numberFormat='0.00E+00'))

mapping_export=mapping.rename(columns={'nace':'NACE','coicop_prefixes':'COICOP basket','proxy_type':'Proxy type','matched_codes':'Code count','fallback':'Empty-basket fallback','source_consumption_bin_tl':'Source consumption (thousand TRY)','ras_multiplier_min':'RAS minimum','ras_multiplier_max':'RAS maximum'})
mapping_export.index.name='Account'
mapping_export['Empty-basket fallback']=mapping_export['Empty-basket fallback'].map({False:'No',True:'Yes'})
def display_selector(selector):
    return '*' if selector=='*' else ' | '.join(p+'*' for p in selector.split('|'))
mapping_export['COICOP basket']=mapping_export['COICOP basket'].map(display_selector)
bridge=frame_spec('Consumption_Bridge',mapping_export,{0:13,1:14,2:62,3:20,4:12,5:18,6:24,7:17,8:17})
bridge['text_columns']=[2]
bridge['formats']=[dict(range='C2:C65',format=dict(wrapText=True)),dict(range='A2:I65',format=dict(rowHeight=48)),dict(range='G2:G65',numberFormat='#,##0.00'),dict(range='H2:I65',numberFormat='0.000000')]
bridge['preview']='A1:F10'

accrows=[]
for a in accounts:
    if a in GROUPS:name=f'{a}: population quintile {GROUPS.index(a)+1} ranked by equivalent income';nace='';role='Household'
    elif a=='HH_REC':name='Zero-sum micro–macro income reconciliation';nace='';role='Reconciliation'
    else:name=str(accounts_source.loc[a,'Name']);nace='' if pd.isna(accounts_source.loc[a,'NACE']) else str(accounts_source.loc[a,'NACE']);role=str(accounts_source.loc[a,'Group'])
    accrows.append(dict(Account=a,NACE=nace,Name=name,Type=role))
account_sheet=frame_spec('Accounts',pd.DataFrame(accrows).set_index('Account'),{0:15,1:16,2:95,3:20})
account_sheet['formats']=[dict(range='C2:C79',format=dict(wrapText=True)),dict(range='A2:D79',format=dict(rowHeight=48))]
account_sheet['preview']='A64:D79'

control=dict(name='Checks',values=[['Account','Row total','Column total','Difference (thousand TRY)','Absolute difference','Result']]+[[a,None,None,None,None,None] for a in accounts],widths={0:22,1:25,2:25,3:21,4:21,5:18},formulas=[],formats=[],preview='A68:F79')
for i,a in enumerate(accounts,2):
    add_formula(control,f'B{i}',f"='SAM_78'!CB{positions[a]+2}",sam.loc[a].sum())
    add_formula(control,f'C{i}',f"='SAM_78'!{xlcol(positions[a]+2)}80",sam[a].sum())
    add_formula(control,f'D{i}',f'=B{i}-C{i}',balance.loc[a,'difference'])
    add_formula(control,f'E{i}',f'=ABS(D{i})',abs(balance.loc[a,'difference']))
    add_formula(control,f'F{i}',f'=IF(E{i}<=0.001,"Pass","Difference detected")')
control['formats']=[dict(range='B2:C79',numberFormat='#,##0.00'),dict(range='D2:E79',numberFormat='0.00E+00')]
control['conditional']=dict(range='E2:E79',formula=.001)

meta_rows=[('Year and unit','2023 macro totals; 2024 HBS distributional shares. SAM: thousand TRY; HBS: TRY.'),('Source SAM','SAM_2023_64_sectors.xlsx; SAM_73!B6:BV78; composite-goods representation.'),('Restricted inputs','External HBS_DATA_DIR; original provider filenames and variable identifiers are retained.'),('Grouping','YILKULGEL/EFB_OECD ranking; FAKTOR * HHB person weights; cumulative weighted midpoints; households indivisible.'),('Mixed-income labour share',str(MIXED_LABOR_SHARE)),('RAS','64 industries by five groups; preserves industry margins and HBS group consumption shares.'),('RAS support prior',str(SUPPORT_PRIOR)),('RAS iterations',str(ras_iterations)),('COICOP notation','011* matches codes beginning with 011; a single * uses the total consumption basket.'),('Tax allocation','Household CTAX follows consumption shares; DTAX follows income shares. Aggregate general-government D.5 is preserved.'),('HH_REC','Zero-sum income reconciliation across groups.'),('Model household income','LAB + CAP + HH_REC receipts - HH_REC payments; subtract DTAX for disposable income.'),('Saving','Reconciled income minus direct taxes minus tax-inclusive consumption; signed values retained.'),('Scaling','TRY to thousand TRY conversion is 0.001. Income and consumption bridges also reconcile year and accounting scope.'),('Income bridge',format(income_bridge_tl_to_sam,'.17g')),('Consumption bridge',format(consumption_bridge_tl_to_sam,'.17g')),('Model data block','SAM_78!B2:CA79; exclude totals. CSV has 78 by 78 entries.'),('HBS source','https://veriportali.tuik.gov.tr/en/press/53957')]
metadata=frame_spec('Metadata',pd.DataFrame(meta_rows,columns=['Topic','Description']).set_index('Topic'),{0:32,1:112})
metadata['formats']=[dict(range=f'B2:B{len(meta_rows)+1}',format=dict(wrapText=True)),dict(range=f'A2:B{len(meta_rows)+1}',format=dict(rowHeight=48))]
metadata['preview']='A1:B12'
job=dict(output=str(OUT/'SAM_2023_64_sectors_5_households.xlsx'),sheets=[main,summary,bridge,account_sheet,control,metadata],expected=expected)
print('Excel: six sheets; main data block SAM_78!B2:CA79.')

## 10. Portable Excel export

The exporter writes numerical arrays and cached formulas directly from Python, without requiring spreadsheet software.

In [ ]:
import sys
for parent in [Path.cwd(),*Path.cwd().parents]:
    if (parent/'tools/public_excel_export.py').is_file():
        sys.path.insert(0,str(parent/'tools'));break
from public_excel_export import export_jobs
export_jobs([job])


## 11. Verify saved outputs

Reopened Excel and CSV values must reproduce the calculated matrix. Account balances, COICOP notation, formula caches and unchanged source inputs are checked.

In [ ]:
path=Path(job['output'])
saved=pd.read_excel(path,sheet_name='SAM_78',index_col=0).iloc[:78,:78]
assert list(saved.index)==list(saved.columns)==accounts
verify('Saved Excel cells',saved,sam,atol=BALANCE_ATOL)  # XLSX serialisation roundoff; 1 TRY tolerance
verify('Saved Excel account balances',saved.sum(axis=1),saved.sum(axis=0))
wb=openpyxl.load_workbook(path,data_only=True,read_only=True)
for ri,s in enumerate(SECTORS,2):
    assert str(wb['Consumption_Bridge'].cell(ri,3).value)==display_selector(SECTOR_BASKETS[s]),('COICOP code format',s)
for test in expected:
    cell=wb[test['sheet']][test['cell']].value
    assert isinstance(cell,(int,float)) and abs(cell-test['value'])<=max(BALANCE_ATOL,abs(test['value'])*1e-12),test
errors=[]
for sh in wb:
    for row in sh:
        for c in row:
            if c.data_type=='e':errors.append([sh.title,c.coordinate,c.value])
assert not errors,errors
wb.close()
csv_path=OUT/'SAM_2023_64_sectors_5_households.csv'
sam.to_csv(csv_path,index_label='Account',encoding='utf-8-sig',float_format='%.17g')
verify('CSV cells',pd.read_csv(csv_path,index_col=0),sam,atol=1e-4)
for entry in manifest['files']:assert sha256((DATA if entry['file'].endswith('.xlsx') else MICRO_DIR)/entry['file'])==entry['sha256']
record=dict(stage='3_household_groups',sam_year=2023,hbs_year=2024,sam_unit='thousand TRY',sam_shape=[78,78],
    group_codes=GROUPS,household_records=len(h),person_records=len(f),consumption_records=len(t),coicop_codes=len(coicop),
    zero_income_households=int((h.YILKULGEL==0).sum()),mixed_labor_share=MIXED_LABOR_SHARE,
    clipped_labor_proxy_households=int((labor_raw<0).sum()),clipped_capital_proxy_households=int((capital_raw<0).sum()),
    ras_support_prior=SUPPORT_PRIOR,ras_iterations=ras_iterations,ras_relative_error=ras_error,
    source_sam_max_imbalance_bin_tl=float((source.sum(axis=1)-source.sum(axis=0)).abs().max()),
    expanded_sam_max_imbalance_bin_tl=float(balance.difference.abs().max()),
    recovery_max_error_bin_tl=float(np.max(abs((collapsed-source).to_numpy()))),
    income_bridge_tl_to_sam=income_bridge_tl_to_sam,consumption_bridge_tl_to_sam=consumption_bridge_tl_to_sam,
    negative_group_savings=profiles.index[profiles.saving0<0].tolist(),
    population_shares=profiles.population_share.to_dict(),income_shares=profiles.income_share.to_dict(),
    consumption_shares=profiles.consumption_share.to_dict(),
    group_calibration=profiles.drop(columns=['equivalent_income_min_tl','equivalent_income_max_tl']).to_dict(orient='index'),
    source_files=manifest['files'],checks=checks,
    output_files=[dict(file=p.name,sha256=sha256(p)) for p in [path,csv_path]],
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,openpyxl=openpyxl.__version__),
    assumptions=dict(institution='Source representative private-sector HOH scope retained; aggregate D.5 unchanged.',
        transfers='HH_REC is zero-net accounting reconciliation, not observed government assistance.',
        final_taxes='Aggregate household CTAX allocated by consumption shares; no sector tax rates imputed.',
        saving='Residual; signed values retained.',sector_scope='All 64 source sectors retained.',
        model_income='LAB + CAP + HH_REC receipts - HH_REC payments; subtract DTAX for disposable income.',
        raw_microdata_copied_to_datas=False,raw_microdata_written_to_outputs=False))
(OUT/'execution_log.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'{len(checks)} numerical checks passed; Excel/CSV verified and execution log saved.')
display(profiles[['population_share','income_share','consumption_share','direct_tax0','saving0']].round(6))

## Interface to the CGE model

Read the 78 accounts without total rows or columns. Household income is labour plus capital plus net HH_REC receipts. HH_REC is an accounting reconciliation. The macro model handles zero-factor/zero-trade branches, signed investment and tax-price relationships explicitly.